# 04 - Evaluation, calibration, languages
Run `python -m src.models.train_model` first so `models/baseline/metrics.json` exists. If `evaluation_status` is `preliminary_synthetic_only`, treat everything below as a pipeline check.

In [ ]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
import matplotlib; matplotlib.rcParams["figure.figsize"] = (6, 3.5)
import matplotlib.pyplot as plt, pandas as pd, numpy as np

In [ ]:
from src.models.model_loader import load_metrics
m = load_metrics(); assert m, "Train first: python -m src.models.train_model"
print(m["model_id"], "| dataset", m["dataset"]["version"], "| evaluation:", m["evaluation_status"]); print(m["evaluation_note"])
pd.DataFrame(m["per_language"]).T

In [ ]:
sel = m["candidates"][m["selected"]]
for split in ("validation", "test"):
    print(split, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in sel[split].items() if k in ("n","precision","recall","f1","f2","accuracy","fpr","fnr","roc_auc","pr_auc")})
pr = sel["test"].get("pr_curve")
if pr: plt.plot(pr["recall"], pr["precision"]); plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Precision-recall (test)")

In [ ]:
from src.models.evaluation import reliability_figure
c = m["calibration"]; print("status:", c["status"], "|", c["note"])
for split in ("validation", "test"):
    print(split, "Brier raw/cal:", round(c[split]["raw"]["brier"], 3), round(c[split]["calibrated"]["brier"], 3), "| ECE raw/cal:", round(c[split]["raw"]["ece"], 3), round(c[split]["calibrated"]["ece"], 3))
reliability_figure(c["validation"]["raw"]["reliability"], c["validation"]["calibrated"]["reliability"], "Reliability (validation)")